In [1]:
import sys
sys.path.append("..")

import json
import joblib
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier

from ml.train_model import load_train_test, evaluate_model, MODEL_DIR, RANDOM_STATE

X_train, X_test, y_train, y_test = load_train_test()
print("X_train:", X_train.shape, "| X_test:", X_test.shape)

X_train: (25927, 20) | X_test: (6482, 20)


In [2]:
with open(MODEL_DIR / "best_model_info.json") as f:
    info = json.load(f)

print(info["model_name"])
print(info["params"])

Random Forest
{'max_depth': 16, 'min_samples_leaf': 2, 'n_estimators': 200, 'random_state': 42}


In [3]:
final_model = RandomForestClassifier(**info["params"], n_jobs=-1)
final_model.fit(X_train, y_train)

scores = evaluate_model(final_model, X_test, y_test)
print({k: round(v, 3) for k, v in scores.items()})

model_path = MODEL_DIR / "credit_risk_model.joblib"
joblib.dump(final_model, model_path, compress=3)
print("Saved:", model_path.name, "| Size MB:", round(model_path.stat().st_size / 1024 / 1024, 1))

{'Accuracy': 0.935, 'Precision': 0.974, 'Recall': 0.721, 'F1': 0.829, 'ROC-AUC': 0.935}
Saved: credit_risk_model.joblib | Size MB: 7.8


In [4]:
model = joblib.load(MODEL_DIR / "credit_risk_model.joblib")

proba = model.predict_proba(X_test)[:, 1]           # Default thavani probability
score = (900 - proba * 600).round().astype(int)

result = pd.DataFrame({"probability": proba, "credit_score": score, "actual_default": y_test.values})
print("Score min / max:", result["credit_score"].min(), "/", result["credit_score"].max())
result.head()

Score min / max: 300 / 900


,probability,credit_score,actual_default
0,0.092624,844,0
1,0.220490,768,1
2,0.034965,879,0
3,0.048663,871,0
4,0.999539,300,1


In [5]:
def risk_level(score):
    if score >= 780:
        return "Low"
    elif score >= 600:
        return "Medium"
    return "High"

result["risk_level"] = result["credit_score"].apply(risk_level)

summary = result.groupby("risk_level").agg(
    applicants=("actual_default", "count"),
    actual_default_rate_pct=("actual_default", lambda s: s.mean() * 100),
).round(1).reindex(["Low", "Medium", "High"])

summary["share_pct"] = (summary["applicants"] / summary["applicants"].sum() * 100).round(1)
summary

,applicants,actual_default_rate_pct,share_pct
risk_level,,,
Low,4923,5.4,75.9
Medium,511,25.6,7.9
High,1048,97.4,16.2


In [6]:
from ml.predict import predict_credit_risk

applicants = {
    "Saaro (high income, nano loan, grade A)": {
        "person_age": 35, "person_income": 1200000, "person_emp_length": 10,
        "loan_amnt": 100000, "loan_int_rate": 7.5, "cb_person_cred_hist_length": 12,
        "loan_grade": "A", "cb_person_default_on_file": "N",
        "person_home_ownership": "OWN", "loan_intent": "EDUCATION",
    },
    "Madhyam (grade C, moderate loan)": {
        "person_age": 28, "person_income": 500000, "person_emp_length": 4,
        "loan_amnt": 200000, "loan_int_rate": 12.0, "cb_person_cred_hist_length": 5,
        "loan_grade": "C", "cb_person_default_on_file": "N",
        "person_home_ownership": "RENT", "loan_intent": "PERSONAL",
    },
    "Kharab (ochi income, motu loan, grade F, pehla default)": {
        "person_age": 24, "person_income": 200000, "person_emp_length": 1,
        "loan_amnt": 150000, "loan_int_rate": 20.0, "cb_person_cred_hist_length": 2,
        "loan_grade": "F", "cb_person_default_on_file": "Y",
        "person_home_ownership": "RENT", "loan_intent": "DEBTCONSOLIDATION",
    },
}

for name, data in applicants.items():
    r = predict_credit_risk(data)
    print(f"{name}\n   -> {r}\n")

Saaro (high income, nano loan, grade A)
   -> {'prediction': 0, 'default_probability': 0.0274, 'credit_score': 884, 'risk_level': 'Low'}

Madhyam (grade C, moderate loan)
   -> {'prediction': 1, 'default_probability': 0.9555, 'credit_score': 327, 'risk_level': 'High'}

Kharab (ochi income, motu loan, grade F, pehla default)
   -> {'prediction': 1, 'default_probability': 0.9825, 'credit_score': 311, 'risk_level': 'High'}



In [7]:
bad = dict(applicants["Saaro (high income, nano loan, grade A)"])
bad["person_income"] = 0
try:
    predict_credit_risk(bad)
except ValueError as e:
    print("Barabar error aavi:", e)

Barabar error aavi: person_income must be greater than 0
